# 02. Star schema and analysis

The Gold tables in notebook 01 each answer one fixed question. For a dashboard that can be sliced any way (by month, state, category), this notebook builds a **star schema** in `workspace.gold`, checks it, answers 5 business questions in SQL and exports the tables for Power BI.

```
                    dim_customer
              (customer_id, city, state)
                        │
   dim_date ──────── fact_orders ─────────── one row per ORDER
 (date, year,           │                    revenue, freight, payment, delivery_days,
  month, weekday)       │                    is_late, review_score, n_items
        │
        └─────────── fact_order_items ─────── dim_product
                     one row per ITEM         (product_id, category, weight)
                     price, freight
```

**Two fact tables, two grains.** `product_id` can't sit on an order-level row (an order can contain several products), and `review_score` can't sit on item rows (a 5-item order would count its review 5 times). Each measure lives at its natural grain, and both facts share the same dimensions.

**Aggregate before joining.** Joining an order to its items *and* its payments multiplies rows (2 items × 2 payments = 4 rows), so every sum doubles. Items and payments are aggregated to one row per order first, then joined.

Runs in the same workspace as notebook 01 (needs its Silver tables and `gold.monthly_sales`).

In [1]:
from pyspark.sql import functions as F

CATALOG = "workspace"

def save(df, layer, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{layer}.{name}")
    print(f"saved {layer}.{name}: {spark.table(f'{CATALOG}.{layer}.{name}').count():,} rows")

def load(layer, name):
    return spark.table(f"{CATALOG}.{layer}.{name}")

# needs the Silver tables from 01_medallion_pipeline
for t in ["orders", "customers", "order_items", "payments", "reviews", "products"]:
    print(t, f"{load('silver', t).count():,}")

orders 99,432
customers 99,441
order_items 112,641
payments 103,874
reviews 98,664
products 32,951


## Dimensions

### `dim_date`
One row per calendar day between the first and last order, including days with no orders, so time charts have no gaps. `sequence` builds the list of dates and `explode` turns it into rows.

In [3]:
orders = load("silver", "orders")
start, end = orders.select(F.min(F.to_date("order_purchase_timestamp")),
                           F.max(F.to_date("order_purchase_timestamp"))).first()
print("from", start, "to", end)

dim_date = (spark.sql(f"SELECT explode(sequence(DATE'{start}', DATE'{end}', INTERVAL 1 DAY)) AS date")
    .withColumn("year",       F.year("date"))
    .withColumn("month",      F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("weekday",    F.date_format("date", "E"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7)))   # Spark: 1 = Sunday, 7 = Saturday

save(dim_date, "gold", "dim_date")      # expected: 774 rows
display(dim_date.limit(10))

from 2016-09-04 to 2018-10-17
saved gold.dim_date: 774 rows
+----------+----+-----+----------+----------+-------+----------+
|date      |year|month|month_name|year_month|weekday|is_weekend|
+----------+----+-----+----------+----------+-------+----------+
|2016-09-04|2016|9    |Sep       |2016-09   |Sun    |true      |
|2016-09-05|2016|9    |Sep       |2016-09   |Mon    |false     |
|2016-09-06|2016|9    |Sep       |2016-09   |Tue    |false     |
|2016-09-07|2016|9    |Sep       |2016-09   |Wed    |false     |
|2016-09-08|2016|9    |Sep       |2016-09   |Thu    |false     |
|2016-09-09|2016|9    |Sep       |2016-09   |Fri    |false     |
|2016-09-10|2016|9    |Sep       |2016-09   |Sat    |true      |
|2016-09-11|2016|9    |Sep       |2016-09   |Sun    |true      |
|2016-09-12|2016|9    |Sep       |2016-09   |Mon    |false     |
|2016-09-13|2016|9    |Sep       |2016-09   |Tue    |false     |
+----------+----+-----+----------+----------+-------+----------+



### `dim_customer`
One row per `customer_id`, with city and state.

In [4]:
dim_customer = load("silver", "customers").select("customer_id", "customer_unique_id", "customer_city", "customer_state")
save(dim_customer, "gold", "dim_customer")      # expected: 99,441

saved gold.dim_customer: 99,441 rows


### `dim_product`
One row per `product_id`, with the English category name as `category`, weight, photo count and `has_listing_info`.

In [5]:
dim_product = load("silver", "products").select(
    "product_id",
    F.col("product_category_name_english").alias("category"),   # rename inside the select
    "product_weight_g",
    "product_photos_qty",
    "has_listing_info")

print("null categories:", dim_product.filter(F.col("category").isNull()).count())   # expected: 0
save(dim_product, "gold", "dim_product")                                            # expected: 32,951

null categories: 0
saved gold.dim_product: 32,951 rows


## Facts

### `fact_orders`: one row per order, all statuses
1. items → one row per order: `n_items`, `items_value`, `freight_value`
2. payments → one row per order: `payment_value`
3. reviews: already one row per order
4. orders **left join** all three, so orders without items, payment or review are kept

`order_date` links to `dim_date`. `is_late_flag` is `is_late` as 1/0, so a BI tool can average it into a late rate.

In [6]:

# 1) items -> one row per order
items_agg = (load("silver", "order_items").groupBy("order_id")
    .agg(F.count("*").alias("n_items"),
         F.round(F.sum("price"), 2).alias("items_value"),
         F.round(F.sum("freight_value"), 2).alias("freight_value")))

# 2) payments -> one row per order (an order can be paid in several parts: card + voucher)
pay_agg = (load("silver", "payments").groupBy("order_id")
    .agg(F.round(F.sum("payment_value"), 2).alias("payment_value")))

# 3) reviews: already one row per order (deduplicated in Silver)
rev = load("silver", "reviews").select("order_id", "review_score")

# 4) orders LEFT JOIN the three (left = never lose an order that has no items/payment/review)
fact_orders = (load("silver", "orders")
    .join(items_agg, "order_id", "left")
    .join(pay_agg,   "order_id", "left")
    .join(rev,       "order_id", "left")
    .withColumn("order_date",   F.to_date("order_purchase_timestamp"))   # key to dim_date
    .withColumn("is_late_flag", F.col("is_late").cast("int"))            # 1/0/null: Power BI can average it
    .select("order_id", "customer_id", "order_date", "order_status", "delivery_days", "is_late", "is_late_flag",
            "n_items", "items_value", "freight_value", "payment_value", "review_score"))

print("rows:", fact_orders.count())                                                          # 99,432
print("unique order_id:", fact_orders.select("order_id").distinct().count())                 # 99,432 (no fan-out)
print("delivered:", fact_orders.filter("order_status = 'delivered'").count())                # 96,469
print("orders with no items:", fact_orders.filter("n_items IS NULL").count())                # 775
print("delivered revenue:", fact_orders.filter("order_status = 'delivered'").agg(F.round(F.sum("items_value"), 2)).first()[0])  # 13,220,069.93
save(fact_orders, "gold", "fact_orders")

rows: 99432
unique order_id: 99432
delivered: 96469
orders with no items: 775
delivered revenue: 13220069.93
saved gold.fact_orders: 99,432 rows


### `fact_order_items`: one row per item
Silver `order_items` joined to `orders` to bring in `customer_id`, `order_status` and `order_date`, so this fact also connects to `dim_customer` and `dim_date`. Each item matches exactly one order, so the join can't fan out: the row count stays at 112,641.

In [7]:
o = load("silver", "orders").select("order_id", "customer_id", "order_status",
                                    F.to_date("order_purchase_timestamp").alias("order_date"))

# each item matches exactly ONE order, so this inner join cannot fan out
fact_order_items = (load("silver", "order_items")
    .join(o, "order_id")
    .select("order_id", "order_item_id", "product_id", "seller_id", "customer_id",
            "order_date", "order_status", "price", "freight_value"))

save(fact_order_items, "gold", "fact_order_items")      # 112,641 = same as Silver order_items

saved gold.fact_order_items: 112,641 rows


## Checks

Row counts, unique keys, no orphan keys between facts and dimensions, and **reconciliation**: delivered revenue is the same whether it's computed from `fact_orders`, `fact_order_items` or `gold.monthly_sales` (R$ 13,220,069.93), even though the three tables were built in different ways.

In [8]:
expected = {"dim_date": 774, "dim_customer": 99_441, "dim_product": 32_951,
            "fact_orders": 99_432, "fact_order_items": 112_641}
for name, n in expected.items():
    got = load("gold", name).count()
    assert got == n, f"{name}: expected {n:,}, got {got:,}"

fo, foi = load("gold", "fact_orders"), load("gold", "fact_order_items")
assert fo.select("order_id").distinct().count() == fo.count(), "fact_orders: order_id not unique (fan-out?)"

# every key in a fact must exist in its dimension (no orphans)
def no_orphans(fact, dim, fact_key, dim_key):
    keys = load("gold", dim).select(F.col(dim_key).alias(fact_key))
    n = fact.join(keys, fact_key, "left_anti").count()
    assert n == 0, f"{n} rows in fact have no match in {dim}"
no_orphans(fo,  "dim_customer", "customer_id", "customer_id")
no_orphans(fo,  "dim_date",     "order_date",  "date")
no_orphans(foi, "dim_product",  "product_id",  "product_id")
no_orphans(foi, "dim_date",     "order_date",  "date")

# reconciliation: revenue three ways
delivered = F.col("order_status") == "delivered"
rev_fact  = fo.filter(delivered).agg(F.sum("items_value")).first()[0]
rev_items = foi.filter(delivered).agg(F.sum("price")).first()[0]
rev_month = load("gold", "monthly_sales").agg(F.sum("revenue")).first()[0]
print(f"fact_orders {rev_fact:,.2f} | fact_order_items {rev_items:,.2f} | monthly_sales {rev_month:,.2f}")
assert abs(rev_fact - 13_220_069.93) < 1
assert abs(rev_fact - rev_items) < 1 and abs(rev_fact - rev_month) < 1

print("star schema checks passed ✅")

fact_orders 13,220,069.93 | fact_order_items 13,220,069.93 | monthly_sales 13,220,069.93
star schema checks passed ✅


## Analysis (SQL)

### 1. Does late delivery hurt reviews?
Average review score for late vs on-time delivered orders.

In [9]:
%sql
-- Q1: late vs on-time reviews (delivered orders)
SELECT is_late,
       ROUND(AVG(review_score), 2) AS avg_review,
       COUNT(*)                    AS n_orders
FROM workspace.gold.fact_orders
WHERE order_status = 'delivered'
GROUP BY is_late

+-------+----------+--------+
|is_late|avg_review|n_orders|
+-------+----------+--------+
|true   |2.56      |7825    |
|false  |4.29      |88644   |
+-------+----------+--------+



> Late orders average **2.56★** vs **4.29★** on time, almost 2 stars lower. Delivery is the biggest lever on ratings we can see, so fixing lateness in the worst states is likely the fastest way to lift the average review (correlation, not proven causation: far-away states could differ in other ways too).

### 2. How long does delivery take? Mean vs median
Mean, median, 90th percentile and maximum `delivery_days` for delivered orders.

In [10]:
%sql
-- Q2: mean vs median delivery days (delivered orders)
SELECT ROUND(AVG(delivery_days), 1)       AS mean_days,
       percentile(delivery_days, 0.5)     AS median_days,
       percentile(delivery_days, 0.9)     AS p90_days,
       MAX(delivery_days)                 AS max_days
FROM workspace.gold.fact_orders
WHERE order_status = 'delivered'

+---------+-----------+--------+--------+
|mean_days|median_days|p90_days|max_days|
+---------+-----------+--------+--------+
|12.1     |10.0       |23.0    |209     |
+---------+-----------+--------+--------+



> Mean **12.1** > median **10** because a few very slow orders (max **209** days) pull the mean up: delivery time is **right-skewed**. On a dashboard card I'd show the **median** (what a typical customer gets) plus the **p90** ("90% arrive within 23 days"), not the mean.

### 3. Where is delivery worst?
Late rate and order count per customer state, top 5.

In [11]:
%sql
-- Q3: worst states by late rate (delivered orders)
SELECT c.customer_state,
       COUNT(*)                            AS n_orders,
       ROUND(AVG(f.is_late_flag) * 100, 1) AS late_pct
FROM workspace.gold.fact_orders f
JOIN workspace.gold.dim_customer c ON f.customer_id = c.customer_id
WHERE f.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY late_pct DESC
LIMIT 5

+--------------+--------+--------+
|customer_state|n_orders|late_pct|
+--------------+--------+--------+
|AL            |397     |23.9    |
|MA            |717     |19.7    |
|PI            |476     |16.0    |
|CE            |1279    |15.3    |
|SE            |335     |15.2    |
+--------------+--------+--------+



> **AL** (Alagoas) is worst at ~**23.9%** late, and the top 5 are all north/north-east states far from most sellers in SP (all 5 are in the north-east). But they're small (335 to 1,279 orders): AL's 23.9% is only ~95 late orders, while **RJ** (12,350 orders, 13.5% late) has ~1,670. Prioritise by *late rate × volume*, not rate alone.

### 4. Cancellation rate
Share of all orders with status `canceled`.

In [12]:
%sql
-- Q4: cancellation rate (ALL orders, so no WHERE)
SELECT ROUND(AVG(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 100, 2) AS cancel_pct,
       SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END)                  AS n_canceled,
       COUNT(*)                                                                    AS n_orders
FROM workspace.gold.fact_orders

+----------+----------+--------+
|cancel_pct|n_canceled|n_orders|
+----------+----------+--------+
|0.63      |625       |99432   |
+----------+----------+--------+



> **0.63%** of orders are canceled, which is small. If it doubled I'd break it down by month, state, category, seller and payment type to see whether one segment drives it (e.g. one seller out of stock, a payment failure, slow approval), then check whether cancellations happen before or after shipping.

### 5. Revenue by month, and incomplete periods

In [13]:
%sql
-- Q5: revenue per month, with a rank (window function over an aggregate)
SELECT d.year_month,
       COUNT(*)                                          AS n_orders,
       ROUND(SUM(f.items_value), 0)                      AS revenue,
       RANK() OVER (ORDER BY SUM(f.items_value) DESC)    AS revenue_rank
FROM workspace.gold.fact_orders f
JOIN workspace.gold.dim_date d ON f.order_date = d.date
WHERE f.order_status = 'delivered'
GROUP BY d.year_month
ORDER BY d.year_month

+----------+--------+--------+------------+
|year_month|n_orders|revenue |revenue_rank|
+----------+--------+--------+------------+
|2016-09   |1       |135.0   |22          |
|2016-10   |265     |40325.0 |21          |
|2016-12   |1       |11.0    |23          |
|2017-01   |750     |111798.0|20          |
|2017-02   |1653    |234223.0|19          |
|2017-03   |2546    |359199.0|17          |
|2017-04   |2303    |340670.0|18          |
|2017-05   |3545    |489159.0|14          |
|2017-06   |3135    |421923.0|16          |
|2017-07   |3872    |481605.0|15          |
|2017-08   |4193    |554700.0|13          |
|2017-09   |4149    |607221.0|12          |
|2017-10   |4478    |648248.0|11          |
|2017-11   |7288    |987648.0|1           |
|2017-12   |5513    |726033.0|10          |
|2018-01   |7069    |924645.0|5           |
|2018-02   |6555    |826437.0|9           |
|2018-03   |7003    |953356.0|4           |
|2018-04   |6798    |973534.0|3           |
|2018-05   |6749    |977545.0|2 

> Peak is **2017-11** (~**988k**): Black Friday. The trap: 2016 has only a handful of orders (Sep 1, Oct 265, Nov missing, Dec 1: the platform was just starting), and after **2018-08** there are no delivered orders at all because the data export ends in Oct 2018 (only undelivered orders remain). These are **incomplete periods, not real drops**. A dashboard should filter to complete months (2017-01 to 2018-08) and say so.

## Export for Power BI

Writes the 5 star-schema tables as CSV files to the volume `workspace.gold.exports` (download them from **Catalog → workspace → gold → Volumes → exports**).
Power BI Desktop can also connect to Databricks directly with the **Azure Databricks** connector.

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.exports")

for t in ["dim_date", "dim_customer", "dim_product", "fact_orders", "fact_order_items"]:
    path = f"/Volumes/{CATALOG}/gold/exports/{t}.csv"
    load("gold", t).toPandas().to_csv(path, index=False)    # small tables, so one file via pandas is fine
    print("wrote", path)

(Databricks-only cell: not run locally)
